In [1]:
%pip install groq python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from groq import Groq

client = Groq(api_key="gsk_LX....")

MODEL_NAME = "llama-3.1-8b-instant"

In [2]:
MODEL_CONFIG = {
    "technical": {
        "system_prompt": """You are a Technical Support Expert.
Provide precise, structured, and code-focused solutions."""
    },

    "billing": {
        "system_prompt": """You are a Billing Support Specialist.
Be empathetic and explain subscription and refund policies clearly."""
    },

    "sales": {
        "system_prompt": """You are a Sales Expert.
Provide persuasive and informative answers about products, features, and pricing.
Encourage potential customers."""
    },

    "tool": {
        "system_prompt": "You are a tool-based expert that fetches live data."
    }
}

In [3]:
def route_prompt(user_input):

    routing_instruction = f"""
Classify the following customer query into ONE of these categories:

technical
billing
sales
tool

Return ONLY the category name.
Do not explain.
Do not add extra words.

Query: {user_input}
"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a strict intent classifier."},
            {"role": "user", "content": routing_instruction}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip().lower().replace(".", "")

In [4]:
def get_bitcoin_price():
    # Mock price (since we are not using real API)
    return "The current price of Bitcoin is $65,000 (mock value)."

In [5]:
def process_request(user_input):
    category = route_prompt(user_input)

    # Tool expert
    if category == "tool":
        answer = get_bitcoin_price()
        return category, answer

    # Safety fallback (important!)
    if category not in MODEL_CONFIG:
        category = "sales"

    expert_config = MODEL_CONFIG[category]

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": expert_config["system_prompt"]},
            {"role": "user", "content": user_input}
        ],
        temperature=0.7
    )

    answer = response.choices[0].message.content

    return category, answer

In [7]:
user_query = input("Enter your query: ")

category, answer = process_request(user_query)

print("\nRouted To:", category)

print("\nResponse:\n", answer)

Enter your query:  I was billed for a service I already cancelled. Can you help me fix this?



Routed To: billing

Response:
 I'm so sorry to hear that you were billed for a service you've already cancelled. That can be frustrating and confusing.

First, let's take a look at your account and see what happened. Can you please provide me with some information about your subscription, such as the date you cancelled and the method you used to cancel (e.g., phone, email, online account)?

Also, can you tell me a bit more about the billing issue you're experiencing? Was it a one-time charge or a recurring charge? How much were you billed?

I'll do my best to help you resolve this issue right away. Our refund and cancellation policies are in place to protect our customers, and I'll work with you to make sure everything is corrected.

In general, our policy is as follows:

* If you cancel your subscription before the next billing cycle, you won't be charged again.
* If you're billed for a service after cancelling, we'll work with you to issue a refund for the incorrect charge.
* If you